# Task 1: Baseline hallucination detection with token probabilities

**Idea.** When a model is unsure, its probability is spread over many next tokens. So an answer made of low-probability tokens (or high-entropy steps) might be a hallucination. We test this on TruthfulQA.

**Steps**
1. Qwen2.5-3B-Instruct answers all 817 questions with greedy decoding (always the most likely token).
2. During that same pass we keep, for every generated token, its log-probability and the entropy of the distribution. Score of an answer = mean log-probability and mean entropy over its tokens.
3. Each answer is labelled accurate or hallucinated by comparing it with TruthfulQA's reference answers.
4. AUC-PR with "hallucinated" as the positive class.

Runtime: T4 GPU, about 15 minutes. Outputs go to `MyDrive/project7/outputs/task1/`.

In [ ]:
# save everything to google drive so task 2 and task 3 can read it later
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/project7/outputs', exist_ok=True)
os.chdir('/content/drive/MyDrive/project7')

!pip install -q bert-score datasets

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

SEED = 42
MODEL = "Qwen/Qwen2.5-3B-Instruct"     # small open model, fits a free T4
SYSTEM = "Answer the question in one short sentence."
MAX_NEW = 64                           # answers are short, 64 tokens is plenty
BATCH = 16
OUT = "outputs/task1"
os.makedirs(OUT, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"

## Data

TruthfulQA, `generation` config: 817 questions, each with lists of correct and incorrect reference answers.

In [ ]:
df = load_dataset("truthfulqa/truthful_qa", "generation", split="validation").to_pandas()
df["id"] = range(len(df))
print(len(df), "questions")
df[["question", "best_answer"]].head()

## Greedy answers + token log-probs and entropy

`output_logits=True` gives the raw logits at every generation step, so we do not need a second forward pass. For each token: log p(token) and entropy H = -sum p log p. Tokens after the end-of-turn token are not counted.

**Decoding settings.** Qwen's `generation_config.json` sets `repetition_penalty=1.05` (plus sampling settings), and `generate()` fills in any setting we don't pass ourselves. A repetition penalty changes the scores before the token is picked, and it also pushes the answer away from words in the question. So we pass `repetition_penalty=1.0`. The check at the end of the cell confirms every generated token is the argmax of the raw logits, i.e. the log-probs we use describe the answer that was actually produced.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to("cuda").eval()
print(model.generation_config)          # shows the defaults shipped with the model
eos = model.generation_config.eos_token_id
eos = set(eos if isinstance(eos, list) else [eos])

rows, n_tok, n_argmax = [], 0, 0
for b in range(0, len(df), BATCH):
    part = df.iloc[b:b + BATCH]
    prompts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
                                       tokenize=False, add_generation_prompt=True) for q in part["question"]]
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, do_sample=False, repetition_penalty=1.0, max_new_tokens=MAX_NEW,
                             return_dict_in_generate=True, output_logits=True, pad_token_id=tok.pad_token_id)
    gen = out.sequences[:, enc["input_ids"].shape[1]:]                     # generated tokens only
    logp = torch.log_softmax(torch.stack(out.logits, dim=1).float(), dim=-1)  # [batch, steps, vocab]
    tok_logp = logp.gather(-1, gen.unsqueeze(-1)).squeeze(-1)
    entropy = -(logp.exp() * logp).nan_to_num().sum(-1)
    for k, i in enumerate(part["id"]):
        ids = gen[k].tolist()
        n = next((t for t, x in enumerate(ids) if x in eos), len(ids))     # stop at the first end token
        n_tok += n
        n_argmax += (logp[k, :n].argmax(-1) == gen[k, :n]).sum().item()
        rows.append({"id": i, "answer": tok.decode(ids[:n], skip_special_tokens=True).strip(),
                     "mean_logprob": tok_logp[k, :n].mean().item(),
                     "mean_entropy": entropy[k, :n].mean().item()})
    print(f"{b + len(part)}/{len(df)}", end="\r")

print(f"\ngreedy check: {n_argmax}/{n_tok} tokens are the argmax of the raw logits")
del model
torch.cuda.empty_cache()

res = df[["id", "question", "best_answer", "category", "type"]].merge(pd.DataFrame(rows), on="id")
res[["question", "answer", "mean_logprob", "mean_entropy"]].head()

## Labels: is the answer accurate?

The dataset document says: accurate if the answer matches or closely paraphrases a correct reference, inaccurate otherwise. We use a similarity rule like the TruthfulQA paper's (they used BLEURT/ROUGE; we use BERTScore): compute the answer's best BERTScore-F1 against the correct references and against the incorrect references. **Accurate if it is closer to a correct one than to any incorrect one.** BERTScore uses `roberta-large`, the same model Task 2 needs, so no extra model is added.

This is automatic and therefore noisy. For example, "No, X" and "Yes, X" look alike to BERTScore, and in TruthfulQA the correct and incorrect references often differ by one word. A small hand check at the end of this notebook measures how often it is right.

In [ ]:
from bert_score import BERTScorer
scorer = BERTScorer(model_type="roberta-large", lang="en", rescale_with_baseline=True)

def best_f1(answers, ref_lists):
    # best BERTScore-F1 of each answer against its list of references
    cands, refs, owner = [], [], []
    for k, (a, rs) in enumerate(zip(answers, ref_lists)):
        for r in rs:
            a_safe = str(a) if str(a).strip() else "empty"
            r_safe = str(r) if str(r).strip() else "empty"
            cands.append(a_safe)
            refs.append(r_safe)
            owner.append(k)

    f1 = scorer.score(cands, refs, batch_size=64)[2].numpy()
    best = np.full(len(answers), -np.inf)
    np.maximum.at(best, owner, f1)
    return best

refs = df.set_index("id").loc[res["id"]]
res["sim_correct"] = best_f1(res["answer"].tolist(), refs["correct_answers"].tolist())
res["sim_incorrect"] = best_f1(res["answer"].tolist(), refs["incorrect_answers"].tolist())
res["hallucinated"] = (res["sim_correct"] <= res["sim_incorrect"]).astype(int)
print(f"hallucinated: {res.hallucinated.mean():.1%} of {len(res)} answers")
del scorer
torch.cuda.empty_cache()

## Results: AUC-PR

Positive class = hallucinated. A random detector gets AUC-PR equal to the share of hallucinated answers, so that is the baseline to beat. Lower log-probability and higher entropy should mean "more likely hallucinated", so we use -mean log-prob and mean entropy as scores.

In [ ]:
ev = res.dropna(subset=["mean_logprob"])
print("empty answers dropped:", len(res) - len(ev))
y = ev["hallucinated"]
scores = {"-mean log-prob": -ev["mean_logprob"], "mean entropy": ev["mean_entropy"]}
table = pd.DataFrame({n: {"AUC-PR": average_precision_score(y, s), "AUROC": roc_auc_score(y, s)}
                      for n, s in scores.items()}).T
table.loc["random"] = [y.mean(), 0.5]
print(table.round(3))
table.to_csv(f"{OUT}/task1_results.csv")

for n, s in scores.items():
    p, r, _ = precision_recall_curve(y, s)
    plt.plot(r, p, label=n)
plt.axhline(y.mean(), ls="--", c="gray", label="random")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Task 1 on TruthfulQA"); plt.legend()
plt.savefig(f"{OUT}/task1_pr.png", dpi=150); plt.show()

## Why is it close to random?

TruthfulQA questions are built to trigger common misconceptions. If the model learned a misconception from web text, it says it confidently, so low uncertainty does not mean the answer is true. We check two things: how confident the model is on hallucinated vs accurate answers, and the result per question type (`Adversarial` questions were selected to fool models).

In [ ]:
conf = np.exp(ev["mean_logprob"])        # mean per-token probability
print("median token probability  hallucinated:", round(conf[y == 1].median(), 3), " accurate:", round(conf[y == 0].median(), 3))
for t, g in ev.groupby("type"):
    print(f"{t}: {len(g)} questions, AUC-PR (entropy) = {average_precision_score(g.hallucinated, g.mean_entropy):.3f}, "
          f"random = {g.hallucinated.mean():.3f}")
print("\nmost confident hallucinations:")
print(ev[y == 1].nsmallest(5, "mean_entropy")[["question", "answer", "mean_entropy"]].to_string())

In [ ]:
res.to_csv(f"{OUT}/task1_features.csv", index=False)

## Hand check of the labels (about 20 minutes, by you)

1. Running this cell writes `outputs/task1/hand_check.csv`: 50 random answers with their reference answers.
2. Open it in Google Sheets, fill `my_label` (1 = matches a correct reference, 0 = wrong) for **all 50 rows**, and save it as `hand_check_filled.csv` in the same folder.
3. Run this cell again: it prints how often the automatic label agrees with you. Put that number in the report.

If an answer changed since you labelled it (for example after re-running generation), that row is skipped and the cell tells you which ids to relabel.

In [ ]:
filled = f"{OUT}/hand_check_filled.csv"
if os.path.exists(filled):
    h = pd.read_csv(filled).dropna(subset=["my_label"])
    m = h[["id", "answer", "my_label"]].merge(res[["id", "answer", "hallucinated"]], on="id", suffixes=("_hand", ""))
    same = m["answer_hand"].fillna("").str.strip() == m["answer"].fillna("").str.strip()
    if (~same).any():
        print("answers changed for ids", m.loc[~same, "id"].tolist(), "-> relabel these")
    m = m[same]
    agree = (m["my_label"] == 1 - m["hallucinated"]).mean()
    print(f"automatic label agrees with my label on {agree:.0%} of {len(m)} answers")
    print(pd.crosstab(m["my_label"], 1 - m["hallucinated"], rownames=["my label"], colnames=["automatic"]))
else:
    s = res.sample(50, random_state=SEED)
    ref = df.set_index("id")
    sheet = pd.DataFrame({"id": s["id"], "question": s["question"], "answer": s["answer"],
                          "correct_answers": [" | ".join(ref.loc[i, "correct_answers"]) for i in s["id"]],
                          "incorrect_answers": [" | ".join(ref.loc[i, "incorrect_answers"]) for i in s["id"]],
                          "my_label": ""})
    sheet.to_csv(f"{OUT}/hand_check.csv", index=False)
    print("wrote", f"{OUT}/hand_check.csv")

In [ ]:
# versions used in this run, for requirements.txt
import importlib.metadata as im
pkgs = ["torch", "transformers", "datasets", "bert-score", "scikit-learn", "numpy", "pandas", "matplotlib"]
lines = [f"{p}=={im.version(p).split('+')[0]}" for p in pkgs]
open("outputs/requirements.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))